In [29]:
import sys
import os
import sys
import os

# Add the project root to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Added {project_root} to sys.path")
sys.path.append(project_root + '/droid_slam')


Added /home/campus.ncl.ac.uk/c4071391/Projects/DROID-SLAM to sys.path


In [31]:
# # copied from demo.py
# # required to instantiate model in the same way so that the model weight dimensions match
# import argparse
# parser = argparse.ArgumentParser()
# parser.add_argument("--imagedir", type=str, help="path to image directory")
# parser.add_argument("--calib", type=str, help="path to calibration file")
# parser.add_argument("--t0", default=0, type=int, help="starting frame")
# parser.add_argument("--stride", default=3, type=int, help="frame stride")

# parser.add_argument("--weights", default="droid.pth")
# parser.add_argument("--buffer", type=int, default=512)
# parser.add_argument("--image_size", default=[240, 320])
# parser.add_argument("--disable_vis", action="store_true")

# parser.add_argument("--beta", type=float, default=0.3, help="weight for translation / rotation components of flow")
# parser.add_argument("--filter_thresh", type=float, default=2.4, help="how much motion before considering new keyframe")
# parser.add_argument("--warmup", type=int, default=8, help="number of warmup frames")
# parser.add_argument("--keyframe_thresh", type=float, default=4.0, help="threshold to create a new keyframe")
# parser.add_argument("--frontend_thresh", type=float, default=16.0, help="add edges between frames whithin this distance")
# parser.add_argument("--frontend_window", type=int, default=25, help="frontend optimization window")
# parser.add_argument("--frontend_radius", type=int, default=2, help="force edges between frames within radius")
# parser.add_argument("--frontend_nms", type=int, default=1, help="non-maximal supression of edges")

# parser.add_argument("--backend_thresh", type=float, default=22.0)
# parser.add_argument("--backend_radius", type=int, default=2)
# parser.add_argument("--backend_nms", type=int, default=3)
# parser.add_argument("--upsample", action="store_true")
# parser.add_argument("--asynchronous", action="store_true")
# parser.add_argument("--frontend_device", type=str, default="cuda")
# parser.add_argument("--backend_device", type=str, default="cuda")

# parser.add_argument("--reconstruction_path", help="path to saved reconstruction")
# args = parser.parse_args([])
# args.stereo = False

In [ ]:
import torch
import onnx
import netron
import collections
from droid_slam.droid_net import DroidNet
# from droid_slam.droid_args import DroidArgs

# DROID is not a standard PyTorch model, so we need its definition.
# Assuming the DROID model class is defined in a 'droid_net.py' file
# as is common in DROID-SLAM implementations.

# --- 1. Load the PyTorch Model ---
pth_model_path = '../droid.pth'
onnx_model_path = '../droid.onnx'

# Instantiate the model
# Adjust arguments if your DroidNet constructor requires them

model = DroidNet()

# Load the weights from the .pth file
# The original DROID weights are often saved with a 'model' key.
state_dict = torch.load(pth_model_path)
print(state_dict.keys())

# The state_dict might be nested or have a 'module.' prefix from DataParallel
# We create a new state_dict to handle these cases
new_state_dict = collections.OrderedDict([
    (k.replace("module.", ""), v) for (k, v) in state_dict.items()])

for k, v in new_state_dict.items():
    print(f'Module {k} : {v.shape}')

new_state_dict["update.weight.2.weight"] = new_state_dict["update.weight.2.weight"][:2]
new_state_dict["update.weight.2.bias"] = new_state_dict["update.weight.2.bias"][:2]
new_state_dict["update.delta.2.weight"] = new_state_dict["update.delta.2.weight"][:2]
new_state_dict["update.delta.2.bias"] = new_state_dict["update.delta.2.bias"][:2]


model.load_state_dict(new_state_dict)


# Set the model to evaluation mode
model.to("cuda:0").eval()


odict_keys(['module.fnet.conv1.weight', 'module.fnet.conv1.bias', 'module.fnet.layer1.0.conv1.weight', 'module.fnet.layer1.0.conv1.bias', 'module.fnet.layer1.0.conv2.weight', 'module.fnet.layer1.0.conv2.bias', 'module.fnet.layer1.1.conv1.weight', 'module.fnet.layer1.1.conv1.bias', 'module.fnet.layer1.1.conv2.weight', 'module.fnet.layer1.1.conv2.bias', 'module.fnet.layer2.0.conv1.weight', 'module.fnet.layer2.0.conv1.bias', 'module.fnet.layer2.0.conv2.weight', 'module.fnet.layer2.0.conv2.bias', 'module.fnet.layer2.0.downsample.0.weight', 'module.fnet.layer2.0.downsample.0.bias', 'module.fnet.layer2.1.conv1.weight', 'module.fnet.layer2.1.conv1.bias', 'module.fnet.layer2.1.conv2.weight', 'module.fnet.layer2.1.conv2.bias', 'module.fnet.layer3.0.conv1.weight', 'module.fnet.layer3.0.conv1.bias', 'module.fnet.layer3.0.conv2.weight', 'module.fnet.layer3.0.conv2.bias', 'module.fnet.layer3.0.downsample.0.weight', 'module.fnet.layer3.0.downsample.0.bias', 'module.fnet.layer3.1.conv1.weight', 'modu

/tmp/ipykernel_1875102/2136071780.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pth_model_path)


DroidNet(
  (fnet): BasicEncoder(
    (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (conv1): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (relu1): ReLU(inplace=True)
    (layer1): Sequential(
      (0): ResidualBlock(
        (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (relu): ReLU(inplace=True)
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      )
      (1): ResidualBlock(
        (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (relu): ReLU(inplace=True)
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affin

In [48]:
# # Creating Dummy args with wrapper
# import torch.nn as nn

# class DroidNetWrapper(nn.Module):
#     def __init__(self):
#         super(self).__init__()
#         self.fnet = BasicEncoder(output_dim=128, norm_fn='instance')
#         self.cnet = BasicEncoder(output_dim=256, norm_fn='none')
#         self.update = UpdateModule()


#     def extract_features(self, images):
#         """ run feature extraction networks """

#         # normalize images
#         images = images[:, :, [2,1,0]] / 255.0
#         mean = torch.as_tensor([0.485, 0.456, 0.406], device=images.device)
#         std = torch.as_tensor([0.229, 0.224, 0.225], device=images.device)
#         images = images.sub_(mean[:, None, None]).div_(std[:, None, None])

#         fmaps = self.fnet(images)
#         net = self.cnet(images)
        
#         net, inp = net.split([128,128], dim=2)
#         net = torch.tanh(net)
#         inp = torch.relu(inp)
#         return fmaps, net, inp


#     def forward(self, Gs, images, disps, intrinsics, graph=None, num_steps=12, fixedp=2):
#         """ Estimates SE3 or Sim3 between pair of frames """

#         u = keyframe_indicies(graph)
#         ii, jj, kk = graph_to_edge_list(graph)

#         ii = ii.to(device=images.device, dtype=torch.long)
#         jj = jj.to(device=images.device, dtype=torch.long)

#         fmaps, net, inp = self.extract_features(images)
#         net, inp = net[:,ii], inp[:,ii]
#         corr_fn = CorrBlock(fmaps[:,ii], fmaps[:,jj], num_levels=4, radius=3)

#         ht, wd = images.shape[-2:]
#         coords0 = pops.coords_grid(ht//8, wd//8, device=images.device)
        
#         coords1, _ = pops.projective_transform(Gs, disps, intrinsics, ii, jj)
#         target = coords1.clone()

#         Gs_list, disp_list, residual_list = [], [], []
#         for step in range(num_steps):
#             Gs = Gs.detach()
#             disps = disps.detach()
#             coords1 = coords1.detach()
#             target = target.detach()

#             # extract motion features
#             corr = corr_fn(coords1)
#             resd = target - coords1
#             flow = coords1 - coords0

#             motion = torch.cat([flow, resd], dim=-1)
#             motion = motion.permute(0,1,4,2,3).clamp(-64.0, 64.0)

#             net, delta, weight, eta, upmask = \
#                 self.update(net, inp, corr, motion, ii, jj)

#             target = coords1 + delta

#             for i in range(2):
#                 Gs, disps = BA(target, weight, eta, Gs, disps, intrinsics, ii, jj, fixedp=2)

#             coords1, valid_mask = pops.projective_transform(Gs, disps, intrinsics, ii, jj)
#             residual = (target - coords1)

#             Gs_list.append(Gs)
#             disp_list.append(upsample_disp(disps, upmask))
#             residual_list.append(valid_mask * residual)


#         return Gs_list, disp_list, residual_list

# DROID typically takes multiple inputs (images, intrinsics, etc.)
# We need to create dummy inputs with the correct shape and type.
# These shapes are based on common usage for DROID-SLAM.
# Please adjust these shapes if your use case is different.

#  b, n, c1, h1, w1 = x.shape
t = 1  # number of time steps
B = 3  # batch size
C = 3  # number of channels
H = 240  # height
W = 320  # width
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# need to provide:
# images
images = torch.randn(t, B, C, H, W).to(device)
# Poses 
Gs = torch.eye(4, device=images.device).view(1, 1, 4, 4).repeat(t, B, 1, 1)
# inverse depths?
disps = torch.ones(t, B, H//8, W//8, device=images.device)
# camera intrinsics
intrinsics = torch.randn(t, B, 4).to(device)
# iterative
graph = {i: [i] for i in range(B)}
num_steps = 12
fixedp = 2

In [49]:

print("Starting ONNX export...")
torch.onnx.export(model,
                  (Gs, images, disps, intrinsics, graph, num_steps, fixedp), # model input
                  onnx_model_path,
                  export_params=True,
                  opset_version=12,
                  do_constant_folding=True,
                  input_names = ['images', 'intrinsics'],
                  output_names = ['outputs'], # adjust if your model has more outputs
                  dynamic_axes={'images' : {0 : 'time'},
                                'intrinsics' : {0 : 'time'}})

print(f"Model has been converted to ONNX and saved at {onnx_model_path}")

# --- 3. Verify and Visualize the ONNX model ---
# Load the ONNX model
onnx_model = onnx.load(onnx_model_path)

# Check that the model is well-formed
onnx.checker.check_model(onnx_model)

print("ONNX model check passed.")
print("Starting Netron visualization server...")

# Visualize the model using Netron
# This will start a web server and open the model in your browser.
netron.start(onnx_model_path)

Starting ONNX export...


AttributeError: 'Tensor' object has no attribute 'inv'